# ENVIRONMENT

In [5]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

In [7]:
import bs4

from langchainhub import Client

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

C:\Users\tongi\AppData\Local\Temp\ipykernel_28148\1637747834.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content" , "post_title" , "post-header")
        )
    ),
)

docs = loader.load()


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

split = text_splitter.split_documents(docs)

In [10]:
vectorstore = Chroma.from_documents(
    documents=split,
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

# INDEX

In [11]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_perspectives|ChatOpenAI(temperature=0)|StrOutputParser()|(lambda x: x.split("\n"))
)

This create mulitiple query around all perspective from a single query or question.

In [12]:
from langchain_core.load import dumps,loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]

question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

C:\Users\tongi\AppData\Local\Temp\ipykernel_28148\1831068274.py:7: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
C:\Users\tongi\AppData\Local\Temp\ipykernel_28148\1831068274.py:7: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


6

After creating multiple queries we find the bunch of chunk for each query then we find the unique chunks across all queries.

In [13]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:
{context}
Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

C:\Users\tongi\AppData\Local\Temp\ipykernel_28148\1831068274.py:7: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


'Task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks. This process can be done through simple prompting, task-specific instructions, or with human inputs. Additionally, LLM agents can utilize external classical planners for long-horizon planning, translating the problem into PDDL, generating a PDDL plan, and translating the plan back into natural language.'

This is the final stage where we connect all thing all get

# RAG - FUSION

In [14]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(temperature=0)
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

This create mulitiple query around all perspective from a single query or question.

In [15]:
from langchain_core.load import dumps,loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

C:\Users\tongi\AppData\Local\Temp\ipykernel_28148\155060614.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


6

In this we have created a function that arrange the chunks to most occuerd to least occured according to number of times of the occurence.

In [16]:
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

C:\Users\tongi\AppData\Local\Temp\ipykernel_28148\155060614.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


'Task decomposition for LLM agents involves breaking down complex tasks into smaller and simpler steps using techniques such as prompting, task-specific instructions, and human inputs. Additionally, there is a distinct approach called LLM+P that involves outsourcing the planning step to an external classical planner, which utilizes the Planning Domain Definition Language (PDDL) to describe the planning problem and generate a plan that is then translated back into natural language.'

This is the final step were we connect all thing and get the respnce.

### 🔄 RAG-Fusion Workflow

```mermaid
graph TD
    A[User asks 1 question] --> B[LLM rewrites it 4 different ways]
    B --> C[Each version hits retriever separately]
    C --> D[4 sets of chunks collected]
    D --> E[RRF scores each chunk based on position]
    E --> F[Chunks ranked highest to lowest score]
    F --> G[Top ranked chunks + original question → LLM]
    G --> H[Final Answer]
```

* **Step 1:** User asks 1 question
* **Step 2:** LLM rewrites it 4 different ways
* **Step 3:** Each version hits retriever separately
* **Step 4:** 4 sets of chunks collected
* **Step 5:** RRF scores each chunk based on position
* **Step 6:** Chunks ranked highest to lowest score
* **Step 7:** Top ranked chunks + original question → LLM
* **Step 8:** Final answer


# DECOMPOSION

In [17]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In this we created a templete that can create sub-querries from one query to get the best responce.

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(temperature=0)

generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})

In this block of code we make sub-queries from single query.

In this we get the answer to each query individually.

In [19]:
answers = []
for q in questions:
    if q.strip():
        clean_q = q
        for prefix in [". ", ") "]:
            if prefix in q[:5]:
                clean_q = q.split(prefix, 1)[-1]
                break
        sub_prompt = ChatPromptTemplate.from_template(
            "Answer the following question based on this context:\n{context}\nQuestion: {question}"
        )
        sub_rag_chain = (
            {"context": retriever, "question": RunnablePassthrough()}
            | sub_prompt
            | llm
            | StrOutputParser()
        )
        answers.append(sub_rag_chain.invoke(clean_q))

def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})

'The main components of an LLM-powered autonomous agent system are planning, memory, and tool use. Planning involves breaking down tasks into smaller subgoals, reflecting on past actions, and refining strategies for future tasks. Memory includes both short-term and long-term memory for learning and information retention. Tool use allows the agent to call external APIs for additional information and leverage other LLMs for complex tasks. These components work together to enable the agent to autonomously plan, learn, reflect, and make decisions, ultimately achieving a higher level of autonomy in problem-solving and decision-making tasks.'

This is the final step here we connect all thing and get the final respose(Here we are using individual approach).

### 🔀 Decomposition (Individual/Parallel Approach) Workflow

```mermaid
graph TD
    A[User asks 1 big question] --> B[LLM breaks it into 3 sub-questions]
    B --> C[Q1 → retrieve → answer1]
    B --> D[Q2 → retrieve → answer2]
    B --> E[Q3 → retrieve → answer3]
    C --> F[All Q&A pairs combined]
    D --> F
    E --> F
    F --> G[Combined context + original question → LLM]
    G --> H[Final Answer]
```

* **Step 1:** User asks 1 big question
* **Step 2:** LLM breaks it into 3 sub-questions
* **Step 3:** Each sub-question is processed individually:
  * Q1 → retrieve → answer1
  * Q2 → retrieve → answer2
  * Q3 → retrieve → answer3
* **Step 4:** All Q&A pairs are combined
* **Step 5:** Combined context + original question → LLM
* **Step 6:** Final answer


In [20]:
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

We create a template such that we can use the reference of previous question_answer to the next one.

In [21]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# llm
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

q_a_pairs = ""
for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

This is the final stage were we connect all thing and get the final response (Here we are using recursive approach).

### 🔁 Decomposition (Recursive/Sequential Approach) Workflow

```mermaid
graph TD
    A[User asks 1 big question] --> B[LLM breaks it into 3 sub-questions]
    B --> C[Q1 → retrieve chunks → LLM answers Q1]
    C --> D[Q2 + Answer1 → retrieve chunks → LLM answers Q2]
    D --> E[Q3 + Answer1 + Answer2 → retrieve chunks → LLM answers Q3]
    E --> F[Final Answer]
```

* **Step 1:** User asks 1 big question
* **Step 2:** LLM breaks it into 3 sub-questions
* **Step 3:** Processes sub-questions sequentially, passing prior QA history:
  * Q1 → retrieve chunks → LLM answers Q1
  * Q2 + Answer1 → retrieve chunks → LLM answers Q2
  * Q3 + Answer1 + Answer2 → retrieve chunks → LLM answers Q3
* **Step 4:** Final answer


# STEP - BACK

In [22]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel's was born in what country?",
        "output": "what is Jan Sindel's personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

Here we have learned `FEW SHOT PROMPTING` .In this we convert the query to more generic and relevent query for the best chance for matching the chunk.

In [23]:
generate_queries_step_back = prompt | ChatOpenAI(temperature=0) | StrOutputParser()
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

'What is the process of breaking down tasks for LLM agents?'

Here we pass on the question with the prompt to openai model.

In [24]:
from langchain_core.runnables import RunnableLambda

response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        "step_back_context": generate_queries_step_back | retriever,
        "question": lambda x: x["question"],
    }
    | response_prompt
    | ChatOpenAI(temperature=0)
    | StrOutputParser()
)

chain.invoke({"question": question})

'Task decomposition for LLM agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps that can be easily handled by the agent. This decomposition can be achieved through various methods such as simple prompting, task-specific instructions, or human inputs. \n\nOne approach to task decomposition involves using LLM with simple prompting, where the agent is guided through the steps of a task by providing prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?". Another method is to provide task-specific instructions tailored to the particular task at hand, such as "Write a story outline" for writing a novel. Additionally, task decomposition can also involve human inputs, where external input is provided to assist the agent in breaking down the task.\n\nIn some cases, a distinct approach known as LLM+P (Liu et al. 2023) is used for task decomposition. This approach involves leveraging an external classical planner to perfo

This is the final stage were we give normal query context , new query context and question and connect all this and get the final response.

### 🔙 Step-Back Workflow

```mermaid
graph TD
    A[User asks specific question] --> B[LLM makes it more generic → step back question]
    B --> C[Original question → retriever → normal context]
    B --> D[Step back question → retriever → step back context]
    C --> E[Both contexts + original question → LLM]
    D --> E
    E --> F[Final Answer]
```

* **Step 1:** User asks specific question
* **Step 2:** LLM makes it more generic → step back question
* **Step 3:** Retrieve context for both questions:
  * Original question → retriever → normal context
  * Step back question → retriever → step back context
* **Step 4:** Both contexts + original question → LLM
* **Step 5:** Final answer


# HYDE

In [25]:
from langchain_core.prompts import ChatMessagePromptTemplate

template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_docs_for_retrieval = (
    prompt_hyde | ChatOpenAI(temperature=0) | StrOutputParser() 
)

question = "What is task decomposition for LLM agents?"
generate_docs_for_retrieval.invoke({"question":question})

'Task decomposition is a fundamental concept in the field of reinforcement learning and artificial intelligence, particularly for Large Language Models (LLMs) agents. Task decomposition refers to the process of breaking down a complex task into smaller, more manageable sub-tasks or components. This approach allows LLM agents to effectively tackle large and complex problems by dividing them into smaller, more easily solvable parts.\n\nBy decomposing a task, LLM agents can focus on solving individual components sequentially or in parallel, leading to more efficient and effective problem-solving strategies. Task decomposition also enables LLM agents to leverage their language understanding capabilities to interpret and generate human-readable instructions for each sub-task, facilitating communication and collaboration with other agents or human users.\n\nOverall, task decomposition plays a crucial role in enhancing the scalability, flexibility, and performance of LLM agents in handling co

Here we try to make a fake response from the query and then generate the real response from fake response.

In [26]:
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

[Document(id='2ef859ed-3cf9-442c-abf9-ec9fbf3418cb', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.\nAnother quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of 

Here we create a hypothetical response.

In [27]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":retrieved_docs,"question":question})

'Task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks. This process can be done through simple prompting, task-specific instructions, or with human inputs. Additionally, LLM agents can utilize external classical planners to do long-horizon planning, translating the problem into Planning Domain Definition Language (PDDL) and generating plans based on existing domain-specific PDDL.'

Here we create the real response from the hypothetical response.

### 🧪 HyDE (Hypothetical Document Embeddings) Workflow

```mermaid
graph TD
    A[User asks question] --> B[LLM generates fake detailed answer]
    B --> C[Fake answer → embedded → search Chroma]
    C --> D[Real matching chunks retrieved]
    D --> E[Real chunks + original question → LLM]
    E --> F[Final Answer]
```

* **Step 1:** User asks question
* **Step 2:** LLM generates fake detailed answer (hypothetical passage)
* **Step 3:** Fake answer → embedded → search Chroma
* **Step 4:** Real matching chunks retrieved
* **Step 5:** Real chunks + original question → LLM
* **Step 6:** Final answer
